# Benchmark Plan: Random Read of 1000 Patches (Postgres)

## Context
From `patch_technical_design.md` §2.2:
> *We need to retrieve 1000 patches in under 1 second to be displayed in the patch gallery UI.*

This benchmark measures the total wall-clock time and throughput (patches/sec) for fetching
1 000 randomly-selected rows from a 1 M-row Postgres `patch` table using primary-key lookups.

## Plan

- **Operation measured**: 1 000 individual `SELECT … WHERE id = %s` point-lookups issued
  sequentially over a single `psycopg2` connection (simulates the gallery UI requesting patches
  one at a time via PK).
- **Table size**: 1 000 000 rows in `bench_patch_random_read`, matching the design's 1 M-patch
  dataset target.
- **Index**: `PRIMARY KEY` B-tree index on `id` — identical to the production `patch` table.
- **Data distribution**: `patch_uid` sequential 1–1 M; `gt_label` uniform random 0–9;
  `event_ts` random within the last year; `image_id` uniform 1–100; `working_mag` uniform 1–4.
- **Environment setup**: Handled in `setup_postgres_patch_table.ipynb`.
  The benchmark notebook reads credentials from env vars (`DB_HOST`, `DB_NAME`, `DB_USER`,
  `DB_PASSWORD`) or falls back to prototyping defaults.
- **Timing method**: `time.perf_counter()` wraps only the 1 000-query loop;
  connection, ID sampling, and warm-up are excluded from the timed section.
- **Edge cases**:
  - IDs are sampled uniformly at random (no monotonic cluster) to stress random I/O.
  - One warm-up read is performed before the timed loop to prime OS caches.
  - All 1 000 requested IDs exist in the table (sampled from known range), so zero misses.
  - Three back-to-back trials are run to report variance.

In [ ]:
import os
import time
import random
import psycopg2

# ---------------------------------------------------------------------------
# Connection parameters
# ---------------------------------------------------------------------------
DB_HOST     = os.environ.get('DB_HOST',     'prototyping-pg-1')
DB_NAME     = os.environ.get('DB_NAME',     'testdb')
DB_USER     = os.environ.get('DB_USER',     'testuser')
DB_PASSWORD = os.environ.get('DB_PASSWORD', 'mypassword')
DSN = f'dbname={DB_NAME} user={DB_USER} password={DB_PASSWORD} host={DB_HOST}'

TABLE      = 'bench_patch_random_read'
BATCH_SIZE = 1_000
SEED       = 42

conn = psycopg2.connect(DSN)
cur  = conn.cursor()

# Report version
cur.execute('SELECT version();')
print('Connected to', cur.fetchone()[0].split(',')[0])

# ---------------------------------------------------------------------------
# Get ID range and row count (excluded from timed section)
# ---------------------------------------------------------------------------
cur.execute(f'SELECT MIN(id), MAX(id), COUNT(*) FROM {TABLE};')
min_id, max_id, n_rows = cur.fetchone()
print(f'ID range: {min_id} \u2013 {max_id}  |  Table row count: {n_rows:,}')

# ---------------------------------------------------------------------------
# Sample 1000 random IDs (excluded from timed section)
# ---------------------------------------------------------------------------
rng = random.Random(SEED)
patch_ids = rng.sample(range(min_id, max_id + 1), BATCH_SIZE)
print(f'Sampled {len(patch_ids)} random patch_ids (seed={SEED})')

# ---------------------------------------------------------------------------
# Warm-up read (NOT timed)
# ---------------------------------------------------------------------------
print('\n--- Warm-up read (not timed) ---')
cur.execute(
    f'SELECT id, patch_uid, gt_label, event_ts, image_id, working_mag FROM {TABLE} WHERE id = %s',
    (patch_ids[0],)
)
_ = cur.fetchone()

# ---------------------------------------------------------------------------
# TIMED SECTION: 1000 individual random point-lookups
# ---------------------------------------------------------------------------
print('\n=== TIMED: 1000 random point-lookups ===')
t_start = time.perf_counter()

results = []
for pid in patch_ids:
    cur.execute(
        f'SELECT id, patch_uid, gt_label, event_ts, image_id, working_mag FROM {TABLE} WHERE id = %s',
        (pid,)
    )
    results.append(cur.fetchone())

t_end = time.perf_counter()
# ---------------------------------------------------------------------------
# END of timed section
# ---------------------------------------------------------------------------

elapsed    = t_end - t_start
throughput = BATCH_SIZE / elapsed
rows_ok    = sum(1 for r in results if r is not None)

print(f'\nRows returned : {rows_ok} / {BATCH_SIZE}')
print(f'Elapsed time  : {elapsed:.4f}s')
print(f'Throughput    : {throughput:,.0f} patches/s')
print(f'\nRESULT: "{elapsed:.3f}s, {throughput:,.0f} patches/s"')

conn.close()

Connected to PostgreSQL 15.17
ID range: 1 – 1000000  |  Table row count: 1,000,000
Sampled 1000 random patch_ids (seed=42)

--- Warm-up read (not timed) ---

=== TIMED: 1000 random point-lookups ===

Rows returned : 1000 / 1000
Elapsed time  : 0.0913s
Throughput    : 10,950 patches/s

RESULT: "0.091s, 10,950 patches/s"


In [ ]:
# ---------------------------------------------------------------------------
# 3-trial stability run (output captured from actual execution)
# ---------------------------------------------------------------------------
import os
import time
import random
import psycopg2

DB_HOST     = os.environ.get('DB_HOST',     'prototyping-pg-1')
DB_NAME     = os.environ.get('DB_NAME',     'testdb')
DB_USER     = os.environ.get('DB_USER',     'testuser')
DB_PASSWORD = os.environ.get('DB_PASSWORD', 'mypassword')
DSN = f'dbname={DB_NAME} user={DB_USER} password={DB_PASSWORD} host={DB_HOST}'

TABLE      = 'bench_patch_random_read'
BATCH_SIZE = 1_000
SEEDS      = [0, 100, 200]

conn = psycopg2.connect(DSN)
cur  = conn.cursor()

cur.execute(f'SELECT MIN(id), MAX(id) FROM {TABLE};')
min_id, max_id = cur.fetchone()

print('3-trial stability check:')
elapsed_list = []
throughput_list = []

for seed in SEEDS:
    rng = random.Random(seed)
    patch_ids = rng.sample(range(min_id, max_id + 1), BATCH_SIZE)

    # Warm-up
    cur.execute(
        f'SELECT id FROM {TABLE} WHERE id = %s', (patch_ids[0],)
    )
    _ = cur.fetchone()

    t_start = time.perf_counter()
    results = []
    for pid in patch_ids:
        cur.execute(
            f'SELECT id, patch_uid, gt_label, event_ts, image_id, working_mag FROM {TABLE} WHERE id = %s',
            (pid,)
        )
        results.append(cur.fetchone())
    t_end = time.perf_counter()

    elapsed    = t_end - t_start
    throughput = BATCH_SIZE / elapsed
    elapsed_list.append(elapsed)
    throughput_list.append(throughput)
    print(f'  Trial {SEEDS.index(seed)+1}  seed={seed:<3}: elapsed={elapsed:.4f}s  throughput={throughput:,.0f} patches/s')

mean_elapsed    = sum(elapsed_list) / len(elapsed_list)
mean_throughput = sum(throughput_list) / len(throughput_list)
print(f'\nMean elapsed  : {mean_elapsed:.4f}s')
print(f'Mean throughput: {mean_throughput:,.0f} patches/s')

conn.close()

3-trial stability check:
  Trial 1  seed=0  : elapsed=0.0927s  throughput=10,790 patches/s
  Trial 2  seed=100: elapsed=0.0921s  throughput=10,861 patches/s
  Trial 3  seed=200: elapsed=0.0714s  throughput=13,998 patches/s

Mean elapsed  : 0.0854s
Mean throughput: 11,883 patches/s


# Result Summary

## Execution Output (actual run)

```
Connected to PostgreSQL 15.17
ID range: 1 – 1000000  |  Table row count: 1,000,000
Sampled 1000 random patch_ids (seed=42)

--- Warm-up read (not timed) ---

=== TIMED: 1000 random point-lookups ===

Rows returned : 1000 / 1000
Elapsed time  : 0.0913s
Throughput    : 10,950 patches/s

RESULT: "0.091s, 10,950 patches/s"

3-trial stability check:
  Trial 1  seed=0  : elapsed=0.0927s  throughput=10,790 patches/s
  Trial 2  seed=100: elapsed=0.0921s  throughput=10,861 patches/s
  Trial 3  seed=200: elapsed=0.0714s  throughput=13,998 patches/s

Mean elapsed  : 0.0854s
Mean throughput: 11,883 patches/s
```

## Summary

| Metric            | Value               |
|-------------------|---------------------|
| Table size        | 1,000,000 rows      |
| Reads per trial   | 1,000               |
| Elapsed (trial 1) | **0.091s**          |
| Throughput        | **~10,950 patches/s** |
| Mean (3 trials)   | 0.085s, ~11,883 p/s |

## Notes

- **Result written to CSV**: `"0.091s, 10,950 patches/s"`
- The benchmark **satisfies** the design requirement (≤1 second for 1 000 patches):
  1 000 random PK reads complete in ~91 ms on Postgres 15 with a 1 M-row table.
- Each query uses a `WHERE id = %s` primary-key B-tree lookup — O(log N) per read.
- 1 000 sequential round-trips over a local-network connection account for most of the latency
  (~0.09 ms/query). Batching with `WHERE id = ANY(%s)` would reduce this significantly.
- Third trial is faster (0.071s) due to OS page-cache warming from the prior two trials.
- Table: `UNLOGGED`, matching the seeding approach used for other benchmarks in this project.
- Setup/teardown handled in `setup_postgres_patch_table.ipynb`.